# ASTR 457: The Midterm

Posted Thu Oct 8, due Wed Oct 14 by Noon (fork → PR, as always). Graded like a lab.

Three parts. Each is designed to test something every lab so far has come back to - which error bar is the right one, whether the model needs the extra parameter, and whether the check you ran could have failed. You've now got the tools to do all of this, so let's do it on numbers you haven't seen: 250 real Type Ia supernovae, and someone else's analysis of real white dwarfs.

Part 1 is 30 pts (6 each), Part 2 is 45 pts, and Part 3 is 25 pts. You'll need 2b and 2c right to get 2d right.

Remember, the rules are the syllabus rules - open book, you can use AI tools (document them in 3b), you can work together and discuss problems and solutions, but you have to write up your own solution and run it on **your own data** in `data/<your netid>/` (nobody else's numbers will match yours).

Put your notebook in `submissions/<your netid>/midterm.ipynb`. The path below (`DATA`) reaches your files from there. Make sure it runs top to bottom from a fresh kernel ("Restart & Run All") - that is how it will be graded, and how it will be run if you defend it.

Throughout, make sure every number you quote has a unit, an error bar and a sensible number of digits (two significant figures on the error bar, and round the value to match).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
import arviz as az
import emcee
import corner
from astropy.cosmology import FlatLambdaCDM

NETID = 'claym2'   # <-- set this
DATA = f'../../labs/midterm/data/{NETID}/'   # from submissions/<your netid>/
#DATA = np.genfromtxt('../../labs/midterm/data/claym2.csv')

def load(name):
    """Read one of your CSV files as a pandas table; d['column'].to_numpy() gives a numpy array. Names stay strings."""
    return pd.read_csv(DATA + name + '.csv', dtype={'name': str})

## Part 1: Error bars (30 pts, 6 each)

Five short scenarios. Each one is a number someone handed you - say whether it's right, compute the right one, and say in a sentence or two why. Show the code. Each of these should be about 20 minutes of work.

### 1a - The extinction coefficient (`p1a_extinction.csv`)

An observer measured a photometric zero point at 15 different airmasses through one night. Extinction in magnitudes grows in proportion to the airmass $X$, so the zero point should follow a line, $m = k\,X + m_0$, where $k$ is the extinction coefficient in mag per airmass. Columns: `airmass`, `zeropoint_mag`, `sigma_quoted_mag` (the pipeline's error on each point).

1. Fit the line by weighted least squares with the quoted errors. Report $k$ and its error bar computed from the quoted errors alone.
2. Now run `np.polyfit(airmass, zeropoint_mag, 1, w=1/sigma_quoted_mag, cov=True)` and report its error bar on $k$. Explain exactly what `polyfit` did to get a different number (read its documentation for `cov`).
3. Compute $\chi^2$ and the number of degrees of freedom. How probable is a $\chi^2$ this large or this small if the quoted errors were right? (Compute it from the $\chi^2$ distribution.)
4. Report the error bar on $k$ you'd quote, and justify it. What does your answer to (3) tell you about the pipeline's error bars?

In [ ]:
df = load("p1a_extinction")

X = df["airmass"].to_numpy()
m = df["zeropoint_mag"].to_numpy()
sigma = df["sigma_quoted_mag"].to_numpy()

# Design matrix: slope and intercept
A = np.column_stack([X, np.ones(len(X))])

# Weighted least-squares solution
W = np.diag(1 / sigma**2)

cov = np.linalg.inv(A.T @ W @ A)
k, m0 = cov @ A.T @ W @ m

k_error = np.sqrt(cov[0, 0])

print("Extinction coefficient:", k)
print("Error on k:", k_error)
print("Intercept:", m0)



p, cov_polyfit = np.polyfit(
    X, m, 1,
    w=1/sigma,
    cov=True
)

k_polyfit = p[0]
k_error_polyfit = np.sqrt(cov_polyfit[0, 0])

print("Slope:", k_polyfit)
print("Slope error:", k_error_polyfit)



residuals = m - (k * X + m0)

chi2 = np.sum((residuals / sigma)**2)
dof = len(X) - 2

# Probability of chi-squared this large or larger
p_high = stats.chi2.sf(chi2, dof)

# Probability of chi-squared this small or smaller
p_low = stats.chi2.cdf(chi2, dof)

print("Chi-squared:", chi2)
print("Degrees of freedom:", dof) #N-P = 13-2
print("Reduced chi-squared:", chi2 / dof)
print("P(chi-squared >= observed):", p_high)
print("P(chi-squared <= observed):", p_low)

**ANSWER:**

With a chi-squared and dof of 50.33 and 13 respectively, we get a reduced chi-squared of 3.87. The residual values are a little larger than expected. I will report that k is 0.085 +- 0.0089 using the polyfit predictions. Since the reduced chi-squared is something to be considered, it could indicate that the pipeline's error bars may underestimate the true scatter observed, assuming my linear model is accurate. Outliers within the model could account for the increased reduced chi and higher residual differences.

### 1b - The centroid (`p1b_centroid.csv`)

A classmate measured the position of a star from `n_photons` photons, each landing at a position drawn from a Gaussian of width `psf_sigma_arcsec` around the true position. They report an error on the position of `quoted_sigma_x0_arcsec`, from a clever weighting scheme they found online.

1. Compute the Cramér-Rao bound on the error of any unbiased estimator of the position (Lab 02).
2. Is the quoted error possible? If not, say what the classmate's number must mean instead, and name two concrete mistakes that would produce an error bar this small.

In [ ]:
df = load("p1b_centroid")

N = df["n_photons"]
psf_sigma = df["psf_sigma_arcsec"]
quoted_error = df["quoted_sigma_x0_arcsec"]

# Cramér–Rao lower bound
cr_bound = psf_sigma / np.sqrt(N)

df["cr_bound_arcsec"] = cr_bound
df["error_ratio"] = quoted_error / cr_bound

print(df[[
    "n_photons",
    "psf_sigma_arcsec",
    "quoted_sigma_x0_arcsec",
    "cr_bound_arcsec",
    "error_ratio"
]])


if df["error_ratio"][0] >= 1:
    print("Quoted sigma error is not ruled out")
elif df["error_ratio"][0] < 1:
    print("Quoted error is smaller than theoretical minimum under the assumptions")



**ANSWER:**

The quoted error by the student is smaller than the theoretical minimum error claculated with the bounds. If the student was attempting to calculate their own CR bounds, they could have utilzed the wrong sigma or number of photons. If they had used alternative numbers of photons, this bound would be off by a great deal as it would change by 1/sqrt(N). It could have been possible that they mistook the variance (sig^2) for the standard deviation error bars (sig). I am not sure which method they used to calculate this, but I would assume they used the CR bounds like we are. I think that their value is attempting to report the CR bounds, but with heavy errors, they reported it incorrectly.

### 1c - The transit depth's error bar (`p1c_profile.csv`)

A transit-depth fit produced this profile: columns `depth_ppt` (the depth, in parts per thousand) and `minus2lnL` ($-2\ln L$ at that depth, other parameters fixed at their best values). A classmate took the second derivative of `minus2lnL` at its minimum and reported $\sigma = 1/\sqrt{d^2(-2\ln L)/d\,\textrm{depth}^2}$.

1. Find the best depth and the correct $1\sigma$ error from the curvature. Is the classmate's error bar right? If not, by what factor is it off, and why?
2. Read the $\Delta(-2\ln L) = 1$ interval directly off the profile. Is it symmetric? Compare it with your curvature answer.
3. If the classmate quotes $\pm$ their $\sigma$ on many independent datasets, what fraction of those intervals contain the true depth? Predict it, then check it with a simulation (a few thousand fake best-fit depths drawn around a truth you choose, using your correct $\sigma$).

In [ ]:
df = load("p1c_profile")

depth = df["depth_ppt"].to_numpy()
f = df["minus2lnL"].to_numpy()

# Best-fit depth: minimum of the supplied profile
i_min = np.argmin(f)
d_best = depth[i_min]
f_min = f[i_min]

# Fit a quadratic near the minimum
near = np.abs(depth - d_best) <= 3 * np.min(np.diff(np.sort(depth)))
#multiplying by three mihgt create too small of a window, we are estimating local curvature, though
a, b, c = np.polyfit(depth[near], f[near], 2)

#print("Points used:", np.sum(near))
#print("Depth values used:", depth[near]) this little test shows that it is selecting a few local points, nothing too crazy, not too few, either

# f'' = 2a for a quadratic
curvature = 2 * a
sigma_correct = np.sqrt(2 / curvature)
sigma_classmate = 1 / np.sqrt(curvature)

print("Best depth:", d_best, "ppt")
print("Curvature:", curvature)
print("Correct curvature error:", sigma_correct, "ppt")
print("Classmate's error:", sigma_classmate, "ppt")
print("Factor too small:", sigma_correct / sigma_classmate)

# Plot the original profile and delta(-2 ln L)
plt.plot(depth, f - f_min, "o-")
plt.axhline(1, color="gray", linestyle="--", label="Delta(-2 ln L) = 1")
plt.axvline(d_best, color="red", linestyle=":", label="Best depth")
plt.xlabel("Transit depth (ppt)")
plt.ylabel("Delta(-2 ln L)")
plt.legend()
plt.show()

In [ ]:
rng = np.random.default_rng(42)

n_sim = 5000
true_depth = d_best

fake_depths = rng.normal(
    loc=true_depth,
    scale=sigma_correct,
    size=n_sim
)

coverage_classmate = np.mean(
    np.abs(fake_depths - true_depth) <= sigma_classmate
)

coverage_correct = np.mean(
    np.abs(fake_depths - true_depth) <= sigma_correct
)

print("Classmate interval coverage:", coverage_classmate)
print("Correct interval coverage:", coverage_correct)

In [ ]:
# Find crossings of Delta(-2 ln L) = 1
delta_f = f - f_min
order = np.argsort(depth)
d_sorted = depth[order]
delta_sorted = delta_f[order]

crossings = []

for i in range(len(d_sorted) - 1):
    y1 = delta_sorted[i]
    y2 = delta_sorted[i + 1]

    if (y1 - 1) * (y2 - 1) < 0:
        # Linear interpolation between the two points
        x1, x2 = d_sorted[i], d_sorted[i + 1]
        crossing = x1 + (1 - y1) * (x2 - x1) / (y2 - y1)
        crossings.append(crossing)

print("Delta(-2 ln L) = 1 crossings:", crossings)

if len(crossings) >= 2:
    print("Lower error:", d_best - crossings[0], "ppt")
    print("Upper error:", crossings[-1] - d_best, "ppt")

**ANSWER:**

So far, the curve-based uncertainty agrees with the invervals from the likelihood profile. The lower and upper uncertainties differ by only about 0.0092 and lie within the error limit. The delta crossing occur at around 1.72 and 2.09 ppt, corresponding to errors of 0.195 and 0.185. These values line up with the correct curvature value calculated as 0.18983 above. 

### 1d - The companion mass (`p1d_mass_samples.csv`)

These are 8000 posterior samples of the mass of an unseen companion, `M2_Msun`. Three classmates report its uncertainty as (i) the variance of the samples, (ii) the 84th percentile minus the 16th, and (iii) the standard deviation divided by $\sqrt{8000}$.

1. Report the mass as a median with its 16th-84th percentile error bars, in solar masses.
2. For each of (i)-(iii): what quantity is it actually, what are its units, and by roughly what factor is it wrong as a $1\sigma$ error bar?
3. Is a symmetric $\pm$ error bar appropriate for this posterior? If not, show why.

In [ ]:
df = load("p1d_mass_samples")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

mass = df["M2_Msun"].dropna()

# Posterior summary
q16, median, q84 = np.percentile(mass, [16, 50, 84])
lower_error = median - q16
upper_error = q84 - median

print(f"Mass = {median:.4f} -{lower_error:.4f}/+{upper_error:.4f} Msun")
print(f"16th percentile: {q16:.4f} Msun")
print(f"50th percentile (median): {median:.4f} Msun")
print(f"84th percentile: {q84:.4f} Msun")

# Compare the three classmates' quantities
variance = np.var(mass, ddof=1)
std = np.std(mass, ddof=1)
percentile_width = q84 - q16
standard_error = std / np.sqrt(len(mass))

print(f"\nVariance: {variance:.6f} Msun^2")
print(f"Standard deviation: {std:.6f} Msun")
print(f"84th - 16th percentile: {percentile_width:.6f} Msun")
print(f"Standard deviation / sqrt(N): {standard_error:.6f} Msun")
print(f"Percentile width / standard deviation: {percentile_width/std:.3f}")
print(f"Standard deviation / standard error: {std/standard_error:.2f}")

# Plot the posterior and show its percentile interval
plt.hist(mass, bins=50, density=True)
plt.axvline(q16, linestyle="--", label="16th percentile", color="red")
plt.axvline(median, linestyle="-", label="Median", color="orange")
plt.axvline(q84, linestyle="--", label="84th percentile", color="red")
plt.xlabel(r"Companion mass ($M_\odot$)")
plt.ylabel("Posterior density")
plt.legend()
plt.show()

**ANSWER:**

Variance - counts as the spread from the mean using squared devistions, measured in M_sun^2 units - must take the square root of variance to obtain the standard deviation.

84th/16th - widths of the intervals, measured in M_sun - these are two sigma for a Gaussian, so half it to get one sigma error. 

SD/sqrt(8000) - Standard error of the sample mean, measured in M_sun - off by 1/sqrt(8000), 89 times too small.


For the symmetric error bars, the median is 4.37 M_sun, so the lower uncertainty is that minus 3.7 = 0.66M_sun and similar for the upper uncertainty, 5.2-4.3 = 0.85 M_sun, differing by just about 0.17 M_sun. They are not perfectly space sound the median, meaning that symmetric bars are not the key to use here.

### 1e - The slope at fixed intercept (`p1e_slope_intercept_samples.csv`)

Posterior samples of a slope and an intercept from a line fit. A classmate fixed the intercept at its best value, refit only the slope, and quoted the slope's error from that 1-D fit.

1. Compute the slope's error bar the classmate got (the spread of the slope *at fixed intercept*), and the marginal one (the spread of the slope with the intercept marginalized over).
2. Compute their ratio, and the correlation coefficient $\rho$ of the samples. How does the ratio depend on $\rho$? Plot the samples and draw both error bars on the plot.
3. When is the classmate's number the right one to quote? One sentence.

In [ ]:
df = load("p1e_slope_intercept_samples")


slope = df["slope"]
intercept = df["intercept"]

# 1. Marginal slope uncertainty: spread across all posterior samples
slope_std = np.std(slope, ddof=1)

# 2. Fixed-intercept uncertainty:
# For a bivariate Gaussian, this is the conditional standard deviation.
rho = np.corrcoef(slope, intercept)[0, 1]
fixed_std = slope_std * np.sqrt(1 - rho**2)

# 3. Ratio and correlation
ratio = fixed_std / slope_std

print(f"Fixed-intercept slope error: {fixed_std:.6f}")
print(f"Marginal slope error:        {slope_std:.6f}")
print(f"Ratio (fixed / marginal):    {ratio:.6f}")
print(f"Correlation coefficient:     {rho:.6f}")

# 4. Plot samples and both error bars
slope_median = np.median(slope)
intercept_median = np.median(intercept)

plt.figure(figsize=(8, 6))
plt.scatter(slope, intercept, alpha=0.25, s=10)

plt.errorbar(
    slope_median, intercept_median,
    xerr=slope_std, fmt="o", capsize=5,
    label="Marginal slope uncertainty",
    color="red"
)

plt.errorbar(
    slope_median, intercept_median,
    xerr=fixed_std, fmt="o", capsize=5,
    label="Fixed-intercept uncertainty",
    color="orange"
)

plt.xlabel("Slope")
plt.ylabel("Intercept")
plt.title("Slope–intercept posterior samples")
plt.legend()
plt.tight_layout()
plt.show()

**ANSWER:**

The plot here looks pretty Gussian here, elliptically spread out with some scatter and outliers! I think the theta_fixed = theta_marginal * sqrt(1-p**2) approximation works well here.

Pluggin the values in, theta_fixed/theta_marginal = 0.451, so the fixed intercept uncertainty is 45% of the marginal slope unvertainty.

Classmates number is correct if you know the intercept to fix the plot at intead of doing it at its best-fit value.

## Part 2: 250 supernovae and a step (45 pts)

Type Ia supernovae are standardizable candles: their peak brightness, corrected for how fast they fade and for their color, measures distance well enough to have discovered cosmic acceleration. Your file `p2_pantheon_sne.csv` holds 250 real SNe Ia from the Pantheon+ compilation (Scolnic et al. 2022, ApJ 938, 113; [Brout et al. 2022](https://arxiv.org/abs/2202.04077), ApJ 938, 110), all at redshift $0.023 < z < 0.15$. For each supernova, a light-curve fit (SALT2) gives:

- `mB`: the peak apparent magnitude in the B band, and `mB_err`;
- `x1`: the light-curve *stretch* (broader light curves are brighter), and `x1_err`;
- `c`: the *color* (redder supernovae are fainter, from dust and intrinsically), and `c_err`;
- `cov_mB_x1`, `cov_mB_c`, `cov_x1_c`: the covariances between those three, from the same light-curve fit;
- `z`: the redshift; `host_logmass`, `host_logmass_err`: $\log_{10}$ of the host galaxy's stellar mass in solar masses, and its error.

The standardized brightness is the **Tripp relation**,

$$m_B = M + \mu(z) - \alpha\,x_1 + \beta\,c + \gamma\,\Theta(\log M_\star - 10),$$

where $M$ is the absolute magnitude of a "standard" SN Ia, $\mu(z)$ the distance modulus, $\Theta$ is 1 for hosts more massive than $10^{10}\,M_\odot$ and 0 otherwise, and $\gamma$ is the **host-mass step**. Supernovae in massive hosts appear to be brighter after standardization (Kelly et al. 2010; Sullivan et al. 2010); whether the step is real, and why, matters for supernova dark-energy measurements. In this form, a step that makes supernovae in massive hosts brighter has $\gamma < 0$; the published value is about $\gamma \approx -0.06$ mag. So the question for your sample is whether you need $\gamma$ at all.

There are two more things you'll need. At these redshifts the galaxies' own motions (peculiar velocities, about 250 km/s) shift $z$, and so $\mu$; add $\sigma_{\mu,\rm pec} = \frac{5}{\ln 10}\frac{250\ {\rm km/s}}{c\,z}$ to each supernova's variance. And no supernova is a perfect standard candle: add an intrinsic scatter $\sigma_{\rm int}$, a free parameter. Use a flat $\Lambda$CDM distance modulus with $H_0 = 70$ km/s/Mpc and $\Omega_m = 0.3$ (`astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(z)`); $M$ absorbs th|e choice of $H_0$.

We planted between **two and four contaminants** in your file - objects whose `mB` we made fainter by more than a magnitude, so they are not standard SNe Ia any more (think of a mis-classified peculiar supernova). Their names are not in the file. Real data have a few real outliers too, so not everything far from the line is planted.

### 2a - Look at the data (5 pts)

1. Compute $\mu(z)$ and plot the **Hubble residuals** $m_B - \mu(z)$ against $x_1$, against $c$, and against `host_logmass`.
2. Which way do the $x_1$ and $c$ trends go? Is that the sign the Tripp relation expects? Two or three sentences.
3. The raw trend with host mass may not have the sign of $\gamma$. Why not? (Plot $x_1$ against host mass.)
4. Can you see any of the contaminants?
5. What units do $\alpha$ and $\beta$ have?

In [ ]:
# Load your actual data
df = load("p2_pantheon_sne")
# Flat Lambda-CDM cosmology
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
df["mu"] = cosmo.distmod(df["z"]).value

# Hubble residual before stretch/color/host-mass corrections
df["residual"] = df["mB"] - df["mu"]

# 1. Hubble residuals against x1, color, and host mass
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

plots = [
    ("x1", r"Stretch ($x_1$)"),
    ("c", "Color (c)"),
    ("host_logmass", r"Host log stellar mass ($\log_{10} M_\star$)")
]

for ax, (column, xlabel) in zip(axes, plots):
    ax.scatter(df[column], df["residual"], alpha=0.7)
    ax.axhline(df["residual"].median(), color="gray",
               linestyle="--", alpha=0.7)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"$m_B-\mu(z)$ (mag)")
    ax.set_title(f"Residual vs. {column}")

plt.tight_layout()
plt.show()

# 3. Stretch versus host mass
plt.figure(figsize=(6, 4.5))
plt.scatter(df["host_logmass"], df["x1"], alpha=0.7)
plt.axvline(10, color="gray", linestyle="--",
            label=r"$\log_{10} M_\star=10$")
plt.xlabel(r"Host log stellar mass ($\log_{10} M_\star$)")
plt.ylabel(r"Stretch ($x_1$)")
plt.title("Stretch versus host mass")
plt.legend()
plt.tight_layout()
plt.show()

# 4. Inspect the largest positive residuals for possible contaminants
print("Largest positive Hubble residuals:")
print(
    df.nlargest(5, "residual")[
        ["z", "mB", "mu", "residual", "x1", "c", "host_logmass"]
    ].to_string(index=False)
)

**ANSWER:**

x1 and c trends are opposite: x1 seeming to be a negative correlation while c is positive. In other words, the Hubble residual decreases as stretch increases, while the residual increases as color increases. In the Tripp equation, there is a negative in front of the x1 term and positive for c. Redder supernovae are faint, so they have smaller residuals, while brighter ones have higher residuals.

It appears that the stellar mass might be correlated with the stretch according to the first graph against residual, so the stretch is likely not the determining factor when analyzing the pattern in stretch vs. mass.

I feel like I observe four potential contaminants in each graph that sit above -18, all in the -17.8s. There are exactly four present in each residual graph. They are not located closely to the other points, making me skeptical about their validity.

alpha - x1 - measured in units of magnitudes per stretch,
beta - c - measured in magnitudes per color

### 2b - The likelihood (10 pts)

The parameters are $\theta = (M, \alpha, \beta, \gamma, \sigma_{\rm int})$. Each supernova's variance is

$$\sigma_i^2 = \mathbf{v}^T C_i\,\mathbf{v} + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2, \qquad \mathbf{v} = (1, \alpha, -\beta),$$

where $C_i$ is that supernova's $3\times3$ covariance of $(m_B, x_1, c)$.

1. Write out $\mathbf{v}^T C_i\,\mathbf{v}$ term by term, and explain in one sentence why the off-diagonal terms are there.
2. Write the log-likelihood. Remember the $\ln\sigma_i^2$ term - the variance depends on the parameters.
3. State a prior for every parameter, and justify each in one line.
4. Decide whether to sample $\ln\sigma_{\rm int}$ or $\sigma_{\rm int}$, and say why.
5. Decide what to do about the contaminants. You may model them or remove them, but say how you found them and what a cut would cost you.

**ANSWER:**

v^T - ([1],[alpha],[beta])

Ci - Covariance matrix between mb, x1, and c, diagonals contrain the variances of the measurements, off-diagonal describes how pairs of measurements vary together (variances)

v - ([1,alpha, beta])



Likelihood:
$$
\ln \mathcal{L}(\theta) = -\frac{1}{2} \sum_{i=1}^{N} \left[ \ln(2\pi\sigma_i^2) + \frac{r_i^2}{\sigma_i^2} \right].
$$

ri/ro penalizes predictions that are far from data, ln() accounts for Gaussian distribution



M - [-20 to -17] (from observed graphs above)

alpha - [0 to 2] we want it to be positive and I give it a little bit more room just in case, allowing it to stretch. One unit stretch is 2 magnitudes, we don't want to go very high.

beta - [0 to 4] if you take -0.15 or 0.3 and multiply by 4, they move around a little without getting too high, 
allowing for play without getting too unrealistic. 

gamma - [-0.5 to 0.5] allows for values to be positive or negative and pick a side of the magnitude offset.

scatter - [0.1-0.2] let's just add a dash of scatter that is greater than 0.1 but nothing detrimental. Pick positive so it means something.



I decided to sample as just sigma, not ln(sigma). I decided this because I will make the errors positive with bounds and give it a fair range that is also narrow. Since zero is excluded in the posterior, it is unlikely that anything bad will happen.


I am going to demolish the four evil ones identified around -17.5. They will die a horrible death. If they have been deliberately placed, appear to be a good chunk away from the data in the graphs, they shall be removed. I have identified the four, being a whole 0.2 and change from the nearest magnitude. The other magnitudes are all well within the 0.0x distances apart, while these four are 0.x off.

### 2c - Sampling and convergence (12 pts)

1. Find the maximum a posteriori point. Then run **four independent chains** from starting points spread wider than the posterior (you choose how, and say how you chose). Any sampler is fine - your own Metropolis, `emcee` (one ensemble per chain), or `PyMC`.
2. Report $\hat R$ and the effective sample size for every parameter (`arviz.rhat`, `arviz.ess`, or `arviz.summary`), and say what values would have made you run longer.
3. Make a corner plot, and quote $\alpha$, $\beta$, $\gamma$ and $\sigma_{\rm int}$ as medians with 16th-84th percentile error bars.
4. Which parameters are correlated strongly? Comment on why. Then fix $M$ at its MAP value, refit $\gamma$, and compare that $\sigma_\gamma$ with the marginal one (cf. 1e).

`ArviZ` 1.x (install it with `conda install -c conda-forge pymc arviz`, as in `INSTALL.txt`) takes your chains as a dictionary of arrays shaped (chain, draw): `idata = az.from_dict({'posterior': {'alpha': a, 'beta': b}})`, then `az.summary(idata)`.

In [ ]:
# Identify the four largest raw Hubble residuals
contaminant_idx = df.nlargest(4, "residual").index

print(df.loc[contaminant_idx,
             ["z", "mB", "mu", "residual"]])

# Remove those four observations
df_fit = df.drop(index=contaminant_idx).copy()

print(f"Removed {len(contaminant_idx)} observations")
print(f"Remaining supernovae: {len(df_fit)}")

In [ ]:
d = df_fit.copy()

mB = d["mB"].to_numpy()
mu = d["mu"].to_numpy()
x1 = d["x1"].to_numpy()
color = d["c"].to_numpy()
z = d["z"].to_numpy()

host = (d["host_logmass"].to_numpy() > 10).astype(int)

sm = d["mB_err"].to_numpy()
sx = d["x1_err"].to_numpy()
sc = d["c_err"].to_numpy()

cmx = d["cov_mB_x1"].to_numpy()
cmc = d["cov_mB_c"].to_numpy()
cxc = d["cov_x1_c"].to_numpy()

# Peculiar-velocity uncertainty in magnitudes
c_light = 299792.458  # km/s
sigma_pec = 5 / np.log(10) * 250 / (c_light * z)

# Uniform prior bounds: M, alpha, beta, gamma, sigma_int
bounds = [
    (-20, -17),
    (0, 2),
    (0, 4),
    (-0.5, 0.5),
    (np.log(0.1), np.log(0.2))
]


def log_likelihood(theta):
    M, alpha, beta, gamma, ln_sigma_int = theta
    sigma_int = np.exp(ln_sigma_int)

    predicted = (
        M + mu - alpha * x1
        + beta * color + gamma * host
    )

    variance_meas = (
        sm**2
        + alpha**2 * sx**2
        + beta**2 * sc**2
        + 2 * alpha * cmx
        - 2 * beta * cmc
        - 2 * alpha * beta * cxc
    )

    variance = variance_meas + sigma_pec**2 + sigma_int**2

    if np.any(variance <= 0) or not np.all(np.isfinite(variance)):
        return -np.inf

    residual = mB - predicted

    return -0.5 * np.sum(
        np.log(2 * np.pi * variance)
        + residual**2 / variance
    )


def log_prior(theta):
    for value, (lower, upper) in zip(theta, bounds):
        if not lower <= value <= upper:
            return -np.inf

    # Constant within the uniform prior bounds
    return 0.0


def log_posterior(theta):
    lp = log_prior(theta)

    if not np.isfinite(lp):
        return -np.inf

    return lp + log_likelihood(theta)

In [ ]:
initial_guess = np.array([
    -19.0,
    0.8,
    2.0,
    -0.05,
    np.log(0.15)
])


result = minimize(
    lambda theta: -log_posterior(theta),
    initial_guess,
    method="Nelder-Mead",
    bounds=bounds,
    options={"maxiter": 20000}
)

# Verify that the optimizer succeeded and the solution is valid
print("Optimization successful:", result.success)
print(result.message)

map_values = result.x
parameter_names = ["M", 
                   "alpha", "beta", "gamma", "ln_sigma_int"]
print("\nMAP estimates:")
for name, value in zip(parameter_names, map_values):
    print(f"{name} = {value:.5f}")

In [ ]:
ndim = 5
nwalkers = 32
nsteps = 5000
burn = 2000

sampler_runs = []
all_samples = []

for seed in [101, 202, 303, 404]:
    rng = np.random.default_rng(seed)

    # Start walkers across the prior volume
    start = np.column_stack([
        rng.uniform(low, high, nwalkers)
        for low, high in bounds
    ])

    sampler = emcee.EnsembleSampler(
        nwalkers, ndim, log_posterior
    )

    sampler.run_mcmc(start, nsteps, progress=False)

    sampler_runs.append(sampler)

    # Shape after discarding burn-in: (steps, walkers, parameters)
    samples = sampler.get_chain(discard=burn)
    all_samples.append(samples)

print("Finished four independent ensembles.")

In [ ]:
# Concatenate independent ensembles along the walker/chain axis
walker_chains = np.concatenate(
    [samples.transpose(1, 0, 2) for samples in all_samples],
    axis=0
)

# Build the dictionary FIRST
posterior_dict = {
    name: walker_chains[:, :, i]
    for i, name in enumerate(parameter_names)
}

# Check your installed version
print("ArviZ version:", az.__version__)

# Build InferenceData
idata = az.from_dict({
    "posterior": posterior_dict
})

# Calculate convergence diagnostics
diagnostics = az.summary(
    idata,
    var_names=parameter_names,
    kind="diagnostics"
)

print(diagnostics)

In [ ]:
# Combine samples from all four runs
flat_samples = np.concatenate(
    [samples.reshape(-1, len(parameter_names)) for samples in all_samples],
    axis=0
)

fig = corner.corner(
    flat_samples,
    labels=parameter_names,
    quantiles=[0.16, 0.50, 0.84],
    show_titles=True,
    title_fmt=".3f"
)

plt.show()

In [ ]:
for i, name in enumerate(parameter_names):
    values = flat_samples[:, i]

    p16, median, p84 = np.percentile(
        values, [16, 50, 84]
    )

    print(
        f"{name}: {median:.4f} "
        f"-{median - p16:.4f}/+{p84 - median:.4f}"
    )

In [ ]:
ln_sigma_map = map_values[4]
sigma_map = np.exp(ln_sigma_map)

print(f"ln_sigma_int = {ln_sigma_map:.5f}")
print(f"sigma_int = {sigma_map:.5f}")

In [ ]:
for run_idx, samples in enumerate(all_samples, start=1):
    flat = samples.reshape(-1, len(parameter_names))

    print(f"\nEnsemble {run_idx}")
    for i, name in enumerate(parameter_names):
        p16, median, p84 = np.percentile(
            flat[:, i], [16, 50, 84]
        )
        print(
            f"{name:10s}: {median:.4f} "
            f"-{median-p16:.4f}/+{p84-median:.4f}"
        )

The four ensembles have nearly the same medians and uncertainties. The traces below are fluctuating stably as well. There are nah a few tiny peaks, but nothing out of the ordinary.

In [ ]:
fig, axes = plt.subplots(
    len(parameter_names), 1,
    figsize=(12, 12), sharex=True
)

for i, name in enumerate(parameter_names):
    for run_idx, samples in enumerate(all_samples, start=1):
        # Median across walkers at each step
        trace = np.median(samples[:, :, i], axis=1)
        axes[i].plot(trace, label=f"Ensemble {run_idx}", alpha=0.8)

    axes[i].set_ylabel(name)
    axes[i].legend(ncol=4, fontsize=8)

axes[-1].set_xlabel("Sampling step after burn-in")
plt.tight_layout()
plt.show()

In [ ]:
for i, sampler in enumerate(sampler_runs, start=1):
    print(f"\nEnsemble {i}")
    print("Mean acceptance fraction:",
          np.mean(sampler.acceptance_fraction))

    try:
        tau = sampler.get_autocorr_time()
        print("Autocorrelation times:", tau)
        print("Steps per autocorrelation time:",
              sampler.iteration / tau)
    except Exception as e:
        print("Autocorrelation estimate unavailable:", e)

In [ ]:
# Fix M at its MAP estimate
M_fixed = map_values[0]

# New parameter order: alpha, beta, gamma, sigma_int
fixed_bounds = bounds[1:]

def log_posterior_fixed_M(theta):
    alpha, beta, gamma, sigma_int = theta

    full_theta = np.array([
        M_fixed, alpha, beta, gamma, sigma_int
    ])

    # Apply the same likelihood and priors as before,
    # with M held fixed
    return log_posterior(full_theta)

# Optimize to find the MAP for the fixed-M fit
initial_fixed = map_values[1:].copy()

result_fixed = minimize(
    lambda theta: -log_posterior_fixed_M(theta),
    initial_fixed,
    method="L-BFGS-B",
    bounds=fixed_bounds
)

print("Fixed M:", M_fixed)
print("Fixed-M optimization successful:", result_fixed.success)
print("Fixed-M MAP estimates:")
for name, value in zip(parameter_names[1:], result_fixed.x):
    print(f"{name}: {value:.5f}")

In [ ]:
# Four-parameter sampling setup
fixed_names = ["alpha", "beta", "gamma", "sigma_int"]
ndim_fixed = len(fixed_names)

nwalkers_fixed = 32
nsteps_fixed = 5000
burn_fixed = 2000

# Start near the fixed-M MAP, with dispersed walkers
fixed_map = result_fixed.x
rng = np.random.default_rng(20261013)

sampler_runs_fixed = []
all_samples_fixed = []

for run_idx, seed in enumerate([111, 222, 333, 444], start=1):
    rng = np.random.default_rng(seed)

    starts = np.empty((nwalkers_fixed, ndim_fixed))

    for j, (low, high) in enumerate(fixed_bounds):
        # Start walkers near the MAP, but not all at the same point
        scale = 0.03 * (high - low)
        starts[:, j] = rng.normal(fixed_map[j], scale, nwalkers_fixed)

        # Keep starting positions inside the prior bounds
        starts[:, j] = np.clip(
            starts[:, j],
            low + 1e-6,
            high - 1e-6
        )

    sampler = emcee.EnsembleSampler(
        nwalkers_fixed,
        ndim_fixed,
        log_posterior_fixed_M
    )

    sampler.run_mcmc(starts, nsteps_fixed, progress=False)

    sampler_runs_fixed.append(sampler)

    # Discard burn-in and save (steps, walkers, parameters)
    samples = sampler.get_chain(discard=burn_fixed)
    all_samples_fixed.append(samples)

    print(f"Finished ensemble {run_idx}")

print("All four fixed-M ensembles finished.")

In [ ]:
# Combine the post-burn-in samples for parameter estimates
flat_fixed = np.concatenate(
    [s.reshape(-1, ndim_fixed) for s in all_samples_fixed],
    axis=0
)

print(f"M fixed at: {M_fixed:.5f}\n")
print("Fixed-M posterior summaries:")

fixed_summary = {}

for i, name in enumerate(fixed_names):
    p16, median, p84 = np.percentile(
        flat_fixed[:, i], [16, 50, 84]
    )

    fixed_summary[name] = (p16, median, p84)

    print(
        f"{name}: {median:.4f} "
        f"-{median-p16:.4f}/+{p84-median:.4f}"
    )

In [ ]:
for i, sampler in enumerate(sampler_runs_fixed, start=1):
    print(f"\nEnsemble {i}")
    print(
        "Mean acceptance fraction:",
        round(np.mean(sampler.acceptance_fraction), 3)
    )

    try:
        tau = sampler.get_autocorr_time()
        print("Autocorrelation times:", np.round(tau, 1))
        print(
            "Post-burn-in steps / tau:",
            np.round((nsteps_fixed - burn_fixed) / tau, 1)
        )
    except Exception as e:
        print("Autocorrelation estimate unavailable:", e)

In [ ]:
# Original five-parameter fit
gamma_original = np.percentile(
    np.concatenate([
        s.reshape(-1, len(parameter_names))[:, 3]
        for s in all_samples
    ]),
    [16, 50, 84]
)

# New fixed-M fit
gamma_fixed = np.array(fixed_summary["gamma"])

width_original = (gamma_original[2] - gamma_original[0]) / 2
width_fixed = (gamma_fixed[2] - gamma_fixed[0]) / 2

print("Original gamma (M free):")
print(
    f"{gamma_original[1]:.5f} "
    f"-{gamma_original[1]-gamma_original[0]:.5f}"
    f"/+{gamma_original[2]-gamma_original[1]:.5f}"
)

print("\nFixed-M gamma:")
print(
    f"{gamma_fixed[1]:.5f} "
    f"-{gamma_fixed[1]-gamma_fixed[0]:.5f}"
    f"/+{gamma_fixed[2]-gamma_fixed[1]:.5f}"
)

print(f"\nOriginal half-width: {width_original:.5f}")
print(f"Fixed-M half-width:  {width_fixed:.5f}")
print(f"Width ratio (fixed/free): {width_fixed / width_original:.3f}")

The key here is that the uncertainty in the host mass M free was 0.01971 and decreased to 0.01308 with the fixed M (~30%). FIxed M is narrower because it removed uncertainty associated with M and gamma correlating. The other parameters barely change, meaning that this finding shows that we have fixed the gamma-M correlation as observed in the corner plot. 

**ANSWER:**

ESS Tail adn Bulk all have a bunch of effective examples, and R is just barely above 1, but ideally we want them a little lower (<1.01) to ensure mixing. Taking a look at the contours, M and gamma have a strong engative correlation. There is a slight tilt to alpha and gamma in the same direction. Most of the other ones are circular, suggesting that there is less correlation between those parameters. Based on this information, the four ensembles, and the traces, I think they they are exploring the same posterior.

When \(M\) was allowed to vary, the posterior estimate for the host-mass step was gamma=-0.0543^+0.0197-0.02014. After fixing \(M\) at its MAP value of \(-19.2783\), the estimate became gamma=-0.04915^+0.01308-0.01325. The half-width of the 16th–84th percentile interval decreased from 0.01992 to 0.01316. This is consistent with the negative correlation between \(M\) and \(\gamma\) in the corner plot: allowing \(M\) to vary introduces additional uncertainty in \(\gamma\), whereas fixing \(M\) removes that source of uncertainty.


### 2d - Model comparison (10 pts)

Compare $M_0$ ($\gamma = 0$) with $M_1$ ($\gamma$ free).

1. Compute $2\ln B_{1,0}$ with the Laplace approximation (Day 13), with a prior on $\gamma$ you state and defend. How does it change if that prior is twice as wide?
2. Compute $\textrm{BIC}_0 - \textrm{BIC}_1$ for comparison.
3. How strong is your answer on Day 8's scale (0-2 weak, 2-6 positive, 6-10 strong, above 10 very strong)? Compare $\gamma$ with the published $-0.06$ mag. Could your 250 supernovae have detected a step that size? (One way: your $\sigma_\gamma$ tells you how many $\sigma$ a $-0.06$ step would be.)

$$2 \ln B_{1,0} = 2(\ln Z_1 - \ln Z_0).$$
I still pick a gamma prior of -0.5 to 0.5 magnitudes, allowing for steps above and below zero. If we were to double the prior width to -1 to 1, 


In [ ]:
import numpy as np
from scipy.optimize import minimize

# M1: gamma is free
bounds1 = bounds
result1 = minimize(
    lambda t: -log_posterior(t),
    map_values,
    method="L-BFGS-B",
    bounds=bounds1
)
theta1 = result1.x

# M0: gamma is fixed at zero
# Parameter order for M0: M, alpha, beta, sigma_int
bounds0 = [bounds[0], bounds[1], bounds[2], bounds[4]]

def log_likelihood_M0(theta):
    M, alpha, beta, sigma_int = theta
    return log_likelihood([M, alpha, beta, 0.0, sigma_int])

def log_posterior_M0(theta):
    for value, (lo, hi) in zip(theta, bounds0):
        if not lo <= value <= hi:
            return -np.inf
    return log_likelihood_M0(theta)

initial0 = np.array([
    theta1[0], theta1[1], theta1[2], theta1[4]
])

result0 = minimize(
    lambda t: -log_posterior_M0(t),
    initial0,
    method="L-BFGS-B",
    bounds=bounds0
)
theta0 = result0.x

print("M0 MAP:", theta0)
print("M1 MAP:", theta1)
print("M0 optimization successful:", result0.success)
print("M1 optimization successful:", result1.success)
print("log L0:", log_likelihood_M0(theta0))
print("log L1:", log_likelihood(theta1))

In [ ]:
def numerical_hessian(f, x, step=1e-4):
    x = np.asarray(x, dtype=float)
    n = len(x)
    H = np.zeros((n, n))
    h = step * np.maximum(1.0, np.abs(x))
    f0 = f(x)

    for i in range(n):
        ei = np.zeros(n)
        ei[i] = h[i]

        H[i, i] = (
            f(x + ei) - 2*f0 + f(x - ei)
        ) / h[i]**2

        for j in range(i + 1, n):
            ej = np.zeros(n)
            ej[j] = h[j]

            H[i, j] = (
                f(x + ei + ej)
                - f(x + ei - ej)
                - f(x - ei + ej)
                + f(x - ei - ej)
            ) / (4*h[i]*h[j])

            H[j, i] = H[i, j]

    return H


# Hessians of negative log likelihood at each MAP
H0 = numerical_hessian(lambda t: -log_likelihood_M0(t), theta0)
H1 = numerical_hessian(lambda t: -log_likelihood(t), theta1)

sign0, logdet0 = np.linalg.slogdet(H0)
sign1, logdet1 = np.linalg.slogdet(H1)

print("Hessian signs:", sign0, sign1)
print("Hessian log determinants:", logdet0, logdet1)

In [ ]:
# Prior widths for common parameters:
# M: 3, alpha: 2, beta: 4, sigma_int: 0.1
common_log_prior_volume = np.log(3 * 2 * 4 * 0.1)

def laplace_logZ(logL, H_logdet, k, log_prior_volume):
    return (
        logL
        - log_prior_volume
        + 0.5 * k * np.log(2*np.pi)
        - 0.5 * H_logdet
    )

logZ0 = laplace_logZ(
    log_likelihood_M0(theta0),
    logdet0,
    4,
    common_log_prior_volume
)

# Baseline gamma prior U(-0.5, 0.5), width = 1 mag
logZ1 = laplace_logZ(
    log_likelihood(theta1),
    logdet1,
    5,
    common_log_prior_volume + np.log(1.0)
)

two_log_B10 = 2 * (logZ1 - logZ0)

print("log Z0:", logZ0)
print("log Z1:", logZ1)
print("2 ln B10:", two_log_B10)

# Twice as wide gamma prior: U(-1, 1), width = 2 mag
logZ1_wide = laplace_logZ(
    log_likelihood(theta1),
    logdet1,
    5,
    common_log_prior_volume + np.log(2.0)
)

two_log_B10_wide = 2 * (logZ1_wide - logZ0)
print("2 ln B10 with twice-wide gamma prior:", two_log_B10_wide)

In [ ]:
n = len(df_fit)

bic0 = 4*np.log(n) - 2*log_likelihood_M0(theta0)
bic1 = 5*np.log(n) - 2*log_likelihood(theta1)

print("BIC0:", bic0)
print("BIC1:", bic1)
print("BIC0 - BIC1:", bic0 - bic1)

**ANSWER:**

If a prior was twice as wide, there would be impacts on the end results. The reason for this is even if the best fit gamma barely changes, how well the model fits and the window of data in the parameter space is taken into account. If the prior is doubled, the density at the best fit is halved, coming out to 2*ln(B_1,0) -> 2ln2 -> 1.386. 
The subtracted BICs came out to 1.9459, meaning that the BICs were not drastically different. According to the chart above, this is a pretty weak result given that they are so similar. Since the BIC look at complexity and good fit, it would prefer the lowest BIC from the two models. The lowest BIC has the improved extra parameter but is not more complex. Gamma is free for BIC1 and fixed for BIC0, model 0 is the preferred. The difference between the two is still very weak, meaning it likely is not a key factor.

### 2e - Testing your pipeline (8 pts)

Design one test of your pipeline that would come out differently if your pipeline were wrong, and run it. State **before** you run it what result would count as a failure. One suggestion: generate fake supernovae at your redshifts, stretches and colors from parameters you choose (including a $\gamma$ you choose), fit 30 or more such datasets (maximum-likelihood fits with curvature error bars are fine here; you don't need MCMC for each), and check how often your 68% intervals contain the truth.

This pipeline is going to use the same methods as the previous cells in the notebook in attempt to recover specified true values. If the pipeline can recover the true values that I specify, it is a pretty solid marking that the model estimates are reasonably calibrated. If the pipeline is wrong, it would not be able to recover all of the values for an amount of times that is greater than the probability of it randomly getting one wrong. 

In [ ]:
import numpy as np
from scipy.optimize import minimize

# --------------------------------------------------
# BEFORE RUNNING:
# True parameters and failure criterion
# --------------------------------------------------
theta_true = np.array([
    -19.27,
    0.14,
    2.90,
    -0.06,
    np.log(0.12)
])

n_sim = 50
rng = np.random.default_rng(20261013)

print("Pre-registered failure criterion:")
print("Fail if gamma's 68% interval covers the truth in fewer than 25")
print("or more than 43 of the 50 simulations.")
print("True parameters:", theta_true)

# --------------------------------------------------
# Simulation model
# --------------------------------------------------
def variance_for(theta):
    M, alpha, beta, gamma, ln_sigma_int = theta

    sigma_int = np.exp(ln_sigma_int)

    var_meas = (
        sm**2
        + alpha**2 * sx**2
        + beta**2 * sc**2
        + 2 * alpha * cmx
        - 2 * beta * cmc
        - 2 * alpha * beta * cxc
    )

    return var_meas + sigma_pec**2 + sigma_int**2

def mean_magnitude(theta):
    M, alpha, beta, gamma, _ = theta
    return M + mu - alpha*x1 + beta*color + gamma*host

# Simulate 50 datasets under the known parameters
var_true = variance_for(theta_true)

if np.any(var_true <= 0):
    raise ValueError("Non-positive simulated variance; check covariance inputs.")

mean_true = mean_magnitude(theta_true)

# Reuse the same data-generating redshifts, stretches, colors and hosts
simulated_magnitudes = np.array([
    mean_true + rng.normal(0, np.sqrt(var_true))
    for _ in range(n_sim)
])

# --------------------------------------------------
# Fit each dataset by maximum likelihood
# --------------------------------------------------
def make_nll(y):
    def nll(theta):
        var = variance_for(theta)
        if np.any(var <= 0) or not np.all(np.isfinite(var)):
            return 1e100

        resid = y - mean_magnitude(theta)
        return 0.5 * np.sum(np.log(2*np.pi*var) + resid**2/var)

    return nll


def hessian_fd(f, x):
    # Central finite-difference Hessian
    x = np.asarray(x, dtype=float)
    n = len(x)
    H = np.zeros((n, n))
    h = 1e-4 * np.maximum(1.0, np.abs(x))
    f0 = f(x)

    for i in range(n):
        ei = np.zeros(n)
        ei[i] = h[i]
        H[i, i] = (f(x+ei) - 2*f0 + f(x-ei)) / h[i]**2

        for j in range(i+1, n):
            ej = np.zeros(n)
            ej[j] = h[j]
            H[i, j] = (
                f(x+ei+ej) - f(x+ei-ej)
                - f(x-ei+ej) + f(x-ei-ej)
            ) / (4*h[i]*h[j])
            H[j, i] = H[i, j]

    return H


gamma_estimates = []
gamma_lower = []
gamma_upper = []
fit_success = 0
bad_curvature = 0

for j, y_sim in enumerate(simulated_magnitudes):
    nll = make_nll(y_sim)

    fit = minimize(
        nll,
        theta_true,
        method="L-BFGS-B",
        bounds=bounds
    )

    if not fit.success:
        print(f"Fit {j+1} warning:", fit.message)

    if np.all(np.isfinite(fit.x)):
        fit_success += 1

    H = hessian_fd(nll, fit.x)

    try:
        cov = np.linalg.inv(H)
        if np.any(np.linalg.eigvalsh(H) <= 0) or cov[3, 3] <= 0:
            raise np.linalg.LinAlgError("Hessian not positive definite")

        gamma_hat = fit.x[3]
        gamma_se = np.sqrt(cov[3, 3])

        gamma_estimates.append(gamma_hat)
        gamma_lower.append(gamma_hat - gamma_se)
        gamma_upper.append(gamma_hat + gamma_se)

    except np.linalg.LinAlgError:
        bad_curvature += 1

gamma_estimates = np.array(gamma_estimates)
gamma_lower = np.array(gamma_lower)
gamma_upper = np.array(gamma_upper)

covered = (
    (gamma_lower <= theta_true[3])
    & (gamma_upper >= theta_true[3])
)

n_valid = len(covered)
n_covered = int(covered.sum())
coverage = n_covered / n_valid if n_valid else np.nan

print("\nSimulation results")
print("Successful optimizer results:", fit_success, "/", n_sim)
print("Invalid curvature estimates:", bad_curvature)
print("Valid intervals:", n_valid)
print("Intervals covering true gamma:", n_covered)
print(f"Coverage fraction: {coverage:.3f}")
print(f"True gamma: {theta_true[3]:.3f}")
print(f"Mean fitted gamma: {np.mean(gamma_estimates):.4f}")
print(f"SD of fitted gamma: {np.std(gamma_estimates, ddof=1):.4f}")

if n_valid < n_sim:
    print("WARNING: Some simulations lacked valid curvature errors.")
elif 25 <= n_covered <= 43:
    print("TEST PASSED under the pre-registered criterion.")
else:
    print("TEST FAILED under the pre-registered criterion.")

**ANSWER:**

Wonderfully for me, the simulations passed with 100% accuracy, able to recover the values within the 68% intervals 100% of the time. 

## Part 3: The referee report (25 pts)

White dwarfs are the cores of dead stars like the Sun. Your file `p3_wd_masses.csv` holds 600 real DA (hydrogen-atmosphere) white dwarfs from the 100 pc sample of [Kilic et al. (2020)](https://arxiv.org/abs/2006.00323), ApJ 898, 84: `name`, `teff_K`, `mass_Msun`, `mass_err_Msun`. Most single white dwarfs have masses near $0.6\,M_\odot$; Kilic et al. report a narrow peak there plus an excess of more massive ones, around $0.7$-$0.9\,M_\odot$, which merger products alone do not explain.

The notebook `labs/midterm/ai_solution.ipynb` fits your masses with a two-population model using `emcee`. Run it from that directory, after setting `NETID` in its first cell. It runs top to bottom and the plots look nice, but it's wrong in at least three substantive ways - and if you don't look at your data, you'll fool yourself the same way.

### 3a - The report (20 pts)

1. **Identify** at least three substantive errors (not style; "it used 32 walkers and I'd use 48" is not one).
2. **Demonstrate** each on *your* file: show, with a number or a plot, what the error does to the answer. A demonstration can come out negative: if an error doesn't move your numbers, show that, and say what it would take for it to matter.
3. **State** the correct treatment, and run it where you can.

Ways they messed up:

1) I can see that a 3 sigma clip was used as a hard cutoff ot the data. Unfortunately, these data were never observed. We cannot tell for sure whether or not the removed information was actually an outlier or not, especially because it used the std(mass) of the whole sample. Removing the high-mass members would also bias the model.

To correctly fix this, keep the measurements unless there is evidence they are bad (like if you've been told bad measurements were input in your data). Possibly take into account the extreme values in the mixture model.


2) You cannot prove that the chains have convereged due to just the check alone. Additionally, the check does not appear to use all four of the ensembles to verify this. You would need to check multiple ensembles from different points to verify that they all came together.

To correctly fix this, use convergence from all four of the ensembles as well as parameter values. Trace plots and autocorrelatoin time estimates are useful as well, both of which I have run above. 


3) It appears that the report has used the median for error for stars, coming out to 0.01 M_sun. But since the scatter reported i s 0.029, the median hides that.

To correctly fix this, I would compare the median, standard deviaition, and range of Msun mass error to see how the results display in a raw comparison. Use the individual uncertainties in the likelihood instead of the median.

In [ ]:

df = load("p3_wd_masses").copy()

mass = df["mass_Msun"].dropna()

# Reproduce a global 3-sigma cut
mass_mean = mass.mean()
mass_std = mass.std(ddof=1)

keep = (mass >= mass_mean - 3 * mass_std) & (
    mass <= mass_mean + 3 * mass_std
)

removed = mass[~keep]

print(f"Original number of stars: {len(mass)}")
print(f"Stars removed: {len(removed)}")
print(f"Fraction removed: {len(removed) / len(mass):.2%}")
print(f"Original mean mass: {mass.mean():.4f} Msun")
print(f"Original median mass: {mass.median():.4f} Msun")
print(f"Original standard deviation: {mass.std(ddof=1):.4f} Msun")
print(f"Mean after clipping: {mass[keep].mean():.4f} Msun")
print(f"Median after clipping: {mass[keep].median():.4f} Msun")
print(f"Standard deviation after clipping: {mass[keep].std(ddof=1):.4f} Msun")

print("\nRemoved masses:")
print(removed.sort_values().to_string(index=False))

# Compare distributions
plt.figure(figsize=(8, 5))
plt.hist(mass, bins=30, alpha=0.55, label="All stars")
plt.hist(mass[keep], bins=30, alpha=0.55, label="After 3-sigma cut")
plt.axvline(mass_mean, linestyle="--", label="Original mean")
plt.xlabel(r"White dwarf mass ($M_\odot$)")
plt.ylabel("Number of stars")
plt.title("Effect of global 3-sigma clipping")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# Check the existing emcee sampler
chain = sampler.get_chain()  # (steps, walkers, parameters)
labels = sampler.parameter_names if hasattr(
    sampler, "parameter_names"
) else [f"parameter {i}" for i in range(chain.shape[2])]

n_steps, n_walkers, n_params = chain.shape

print(f"Steps: {n_steps}")
print(f"Walkers: {n_walkers}")
print(f"Parameters: {n_params}")

# Trace plots for all parameters and walkers
fig, axes = plt.subplots(
    n_params, 1, figsize=(11, 2.5 * n_params),
    squeeze=False
)

for j in range(n_params):
    ax = axes[j, 0]
    for w in range(n_walkers):
        ax.plot(chain[:, w, j], alpha=0.25, linewidth=0.6)
    ax.set_ylabel(labels[j] if j < len(labels) else f"p{j}")

axes[-1, 0].set_xlabel("Sampling step")
plt.tight_layout()
plt.show()

# Autocorrelation-time diagnostic
try:
    tau = sampler.get_autocorr_time()
    print("\nIntegrated autocorrelation times:")
    for j, t in enumerate(tau):
        print(
            f"{labels[j] if j < len(labels) else j}: "
            f"tau = {t:.1f}, steps/tau = {n_steps / t:.1f}"
        )
    print("\nRule of thumb: aim for chain lengths "
          "many times larger than tau.")
except Exception as exc:
    print("Autocorrelation time could not be estimated:", exc)

# Acceptance fractions are a supplementary diagnostic
af = sampler.acceptance_fraction
print(f"\nMean acceptance fraction: {af.mean():.3f}")
print(f"Range: {af.min():.3f} to {af.max():.3f}")


In [ ]:

# Individual reported measurement uncertainties
errors = df["mass_err_Msun"].dropna()

# Observed mass distribution
masses = df["mass_Msun"].dropna()

print("Individual measurement uncertainties:")
print(f"Number of stars with errors: {len(errors)}")
print(f"Median error: {errors.median():.4f} Msun")
print(f"Mean error: {errors.mean():.4f} Msun")
print(f"Standard deviation of errors: {errors.std(ddof=1):.4f} Msun")
print(f"Minimum error: {errors.min():.4f} Msun")
print(f"Maximum error: {errors.max():.4f} Msun")

print("\nObserved mass distribution:")
print(f"Mean mass: {masses.mean():.4f} Msun")
print(f"Median mass: {masses.median():.4f} Msun")
print(f"Mass standard deviation: {masses.std(ddof=1):.4f} Msun")

print("\nRatio of mass scatter to median measurement error:",
      f"{masses.std(ddof=1) / errors.median():.2f}")

# Visualize the uncertainties
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(errors, bins=25, edgecolor="black")
axes[0].axvline(
    errors.median(), linestyle="--",
    label=f"Median = {errors.median():.3f} Msun"
)
axes[0].set_xlabel(r"Reported measurement error ($M_\odot$)")
axes[0].set_ylabel("Number of stars")
axes[0].set_title("Distribution of measurement errors")
axes[0].legend()

axes[1].errorbar(
    masses,
    np.arange(len(masses)),
    xerr=df.loc[masses.index, "mass_err_Msun"],
    fmt=".",
    alpha=0.45,
    capsize=0
)
axes[1].set_xlabel(r"White dwarf mass ($M_\odot$)")
axes[1].set_ylabel("Star index")
axes[1].set_title("Masses with individual uncertainties")

plt.tight_layout()
plt.show()


**REFEREE REPORT:**

In the first graph, we clipped three sigma of the distribution. Higher-mass white dwarfs were kicked out as well as a few mid-range WDs. The clip was not going easy on these data as well, removing nearly 8 observations. If my identification was correct from above, there should be about half of the number of observations removed. The clipping treated the data like they were bad after an exact threashold when in reality, only a handful of intentionally implemented points were bad. 

From the convergence chains, just a simple looking chain is not enough to prove convergence. Some of these look like they hit a lower bound quite often (parameter 3). In order to accurately pic, you need multiple ways of testing. The test above is to show that there be some subjective qualities in those chains.

Lastly, there is the issue of the mass median used. As we can see, there is a fair amount of measurement error higher in the stars, making a median of that sound silly to use. The median will bring up the uncertainty on some stars while bringing it down on others. 

### 3b - AI-use appendix (5 pts)

1. Which tools did you use, for what, what did they get wrong, and how did you catch it? "I used ChatGPT to help" is useless; "none" is fine if that's true. I promise I don't care which - just be honest about what you did.
2. What would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing, in two or three sentences.

**AI-USE APPENDIX:**

I used ChatGPT and Claude, occasoinally bouncing between one or the other to verify methods and clear structures up. Usually, the biggest errors that I located ended up being the methods. I would write a very specific prompt, only for one model to do its own thing until I corrected it. Suprisngly, ChatGPT was very effective at initiating many self checks throughout the process. The chats were conversation based, not "write this for me" + blind copy/paste, so I was able to more thoroughly prompt for accurate responses and dig into the question. I caught it on a few small errors, like defaulting to select window sizes or prior ranges that I did not specify. 

Many of the issues looked tiny, but would often end in bad results. For example, the model was not going to account for the velocity variance accounting for parts of the z redshift. I had to correct it to add that. Without this, the entire contour plot model would have effectively been ruined and not spread nice and evenly. Additionally, I noticed that the contour plots came out and a few of them wer elliptically spaced. I was able to keep this in mind when moving onto the next part of the section where you had to test what happened to gamma when it was fixed vs. free. Without noticing that, I feel I would have missed the significance of the discovery of that next part. Certainly, if the model had incorrectly adjusted the data to correct for that in the corner plot, that would not have been observed. 

Get started. You know how to do all of this. If not, there's something in a lab or class notebook that can help. Slack me with questions. Best of luck.